# ⏳ Session Memory with a TTL Store

LangGraph has no built-in "session" concept the way it has a checkpointer for threads — but its
cross-thread **`Store`** (LangGraph's mechanism for memory that spans threads) supports a native
**TTL** (time-to-live), which is exactly the primitive session memory needs: state that lives
for *one visit*, not forever.

This notebook builds a LangGraph agent whose node reads session-scoped preferences
through `SqliteStore` configured with a TTL, and shows precisely how — and how imperfectly —
`SqliteStore` enforces that expiry.

## Learning Objectives
In this notebook, you will learn:
1. **A store with a TTL** - configure `SqliteStore` with `TTLConfig` for session-scoped memory
2. **A session-aware node** - key memory by `session_id`, not `thread_id`
3. **Session scope** - a new conversation in the same visit keeps the preference; a new visit starts without it
4. **How expiry really works** - on `SqliteStore`, an expired item stays readable until a sweep deletes it

## Prerequisites
- `01_cross_thread_memory_store.ipynb` (the Store, and the `Runtime` + `context` pattern) and module 02's
  `01_memory_and_conversational_agent.ipynb` (thread-scoped short-term memory, which session memory spans)
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: answers under the current session's instruction
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## ⏱️ Part 1: A Store With a TTL

`SqliteStore` is LangGraph's long-term (cross-thread) memory backend. Passing a `TTLConfig`
turns it into a session store: every item written gets an expiry, and (if `refresh_on_read` is
set) a read slides the expiry forward — the same "touch on activity" behavior a web session
cookie has.

`default_ttl` is in **minutes**: 30 here, a realistic visit. Part 4 shortens one item's TTL to
a few seconds, so the expiry demo doesn't require actually waiting 30 minutes.

`omit_expired=True` asks the store to hide expired items on read — *if the backend supports it*.
`PostgresStore` does. `SqliteStore` (`langgraph-checkpoint-sqlite` 3.1.1, the version pinned here)
never reads the option; Part 4 shows what that means.

In [ ]:
# ============================================================================
# PART 1: A SESSION STORE WITH A TTL
# ============================================================================
import os
import sqlite3

from langgraph.store.base import TTLConfig
from langgraph.store.sqlite import SqliteStore

DB_PATH = "session_memory.db"
RESET_DB = True  # False keeps memories from earlier runs
if RESET_DB and os.path.exists(DB_PATH):
    os.remove(DB_PATH)

# isolation_level=None hands transaction control to the store itself -- required for
# SqliteStore's own BEGIN/COMMIT handling to work correctly.
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)

session_store = SqliteStore(
    conn,
    ttl=TTLConfig(
        default_ttl=30,               # minutes, not seconds: one visit
        refresh_on_read=False,        # fixed-length visits; True gives sliding expiry
        omit_expired=True,            # honoured by PostgresStore; SqliteStore ignores it (Part 4)
        sweep_interval_minutes=None,  # we sweep by hand in Part 4 instead of a background thread
    ),
)
session_store.setup()
print(f"✅ Session store ready in {DB_PATH} (items expire 30 minutes after they are written)")

---
## 🧭 Part 2: A Session-Aware Node

The node reads a session preference (e.g. "reply in French for this visit") from the store
*before* calling the LLM, using `session_id` — not `thread_id` — as the store key. This is what
separates session memory from short-term memory: it survives a brand-new conversation as long as
the `session_id` is the same and the session hasn't expired.

As in `01_cross_thread_memory_store.ipynb`, the node gets the store and the trusted id through its
`runtime`; here the context carries a `session_id`, which your server issues when the visit starts.

In [ ]:
# ============================================================================
# PART 2: A SESSION-AWARE AGENT NODE
# ============================================================================
from dataclasses import dataclass

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.runtime import Runtime


@dataclass
class Context:
    session_id: str  # trusted: issued by your server when the visit starts


def agent_node(state: MessagesState, runtime: Runtime[Context]) -> dict:
    pref = runtime.store.get(("session", runtime.context.session_id), "reply_style")
    style_note = pref.value["instruction"] if pref else "Respond normally."

    system = SystemMessage(f"Session instruction for this visit only: {style_note}")
    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("agent", agent_node)
builder.add_edge(START, "agent")
builder.add_edge("agent", END)

# No checkpointer: every call is a brand-new conversation (what a new thread would be), so
# whatever carries over between calls must come from the STORE, not from a thread's history.
session_agent = builder.compile(store=session_store)


def ask(text: str, session_id: str) -> str:
    result = session_agent.invoke({"messages": [HumanMessage(text)]}, context=Context(session_id=session_id))
    return result["messages"][-1].content


print("✅ Session-aware agent compiled")

---
## 🔁 Part 3: Session Scope

### 3.1 Same Visit, New Conversation

The app writes a mid-visit preference — say the user flipped a language toggle — and the store
stamps it with the 30-minute TTL. A brand-new conversation under the **same** `session_id`
still picks it up: session memory is keyed by the visit, not the conversation.

In [ ]:
# ============================================================================
# PART 3.1: THE PREFERENCE SURVIVES A NEW CONVERSATION IN THE SAME VISIT
# ============================================================================
SESSION_ID = "session-alice-visit-1"

session_store.put(
    ("session", SESSION_ID),
    "reply_style",
    {"instruction": "Reply only in French for the rest of this visit."},
)

print("conversation 1:", ask("Say hello.", SESSION_ID))
print("conversation 2 (new conversation, SAME session):", ask("Say hello again.", SESSION_ID))

### 3.2 A Different Session Sees Nothing

Session scope means a new `session_id` — a new visit, even for the same person — starts with no
preference.

In [ ]:
# ============================================================================
# PART 3.2: A NEW SESSION IS ISOLATED
# ============================================================================
print("new session (no French instruction carried over):", ask("Say hello.", "session-alice-visit-2"))

---
## ⌛ Part 4: How Expiry Really Works

### 4.1 `get()` Still Returns an Expired Item

This is the gotcha worth seeing with your own eyes rather than taking on faith. `SqliteStore` writes
an `expires_at` time for every item with a TTL, but its `get()` and `search()` never look at it, and
it ignores `omit_expired`. An expired item stays readable until a **sweep** deletes it —
`sweep_ttl()`, or the background thread that `start_ttl_sweeper()` / `sweep_interval_minutes` starts.
`PostgresStore` with `omit_expired=True` filters expired rows on every read instead.

The cell rewrites the preference with a 3-second TTL (`put(..., ttl=...)` overrides `default_ttl`
for one item), waits past it, and reads it back.

In [ ]:
# ============================================================================
# PART 4.1: AN EXPIRED ITEM IS STILL READABLE UNTIL A SWEEP (SqliteStore)
# ============================================================================
import time
from datetime import datetime, timezone

DEMO_TTL_MINUTES = 0.05  # 3 seconds
session_store.put(
    ("session", SESSION_ID),
    "reply_style",
    {"instruction": "Reply only in French for the rest of this visit."},
    ttl=DEMO_TTL_MINUTES,
)
time.sleep(4)  # longer than the 3-second TTL

expired = session_store.get(("session", SESSION_ID), "reply_style")
if expired is None:
    print("get() hid the expired item: this store version honours omit_expired")
else:
    written_at = expired.updated_at.replace(tzinfo=timezone.utc)  # SQLite stores UTC without a zone
    age = (datetime.now(timezone.utc) - written_at).total_seconds()
    print(f"after the TTL, before a sweep: get() returned {expired.value}")
    print(f"  written {age:.0f} s ago with a {DEMO_TTL_MINUTES * 60:.0f} s TTL -- omit_expired=True did not hide it")

swept = session_store.sweep_ttl()
print(f"\nsweep_ttl() deleted {swept} expired row(s)")
print("after the sweep: get() returned", session_store.get(("session", SESSION_ID), "reply_style"))

### 4.2 The Agent After the Sweep

Once swept, the session-aware node falls back to its default behavior — exactly like a session
cookie that's gone.

In [ ]:
# ============================================================================
# PART 4.2: AGENT BEHAVIOR AFTER EXPIRY
# ============================================================================
print("same session after the sweep:", ask("Say hello.", SESSION_ID))

---
## 📝 Summary

In this notebook, we learned:

### 1. A store with a TTL
- Session memory in LangGraph = a `Store` (here, `SqliteStore`) configured with `TTLConfig`
- `default_ttl` is in **minutes**; `put(..., ttl=...)` overrides it for one item

### 2. A session-aware node
- The node reads `runtime.store` and keys the item by `runtime.context.session_id`, not `thread_id`

### 3. Session scope
- A new conversation in the same visit keeps the preference; a new visit starts without it

### 4. How expiry really works
- On `SqliteStore`, an expired item stays readable until `sweep_ttl()` (or a background sweeper) deletes it;
  `omit_expired=True` only hides expired items on backends that support it, such as `PostgresStore`

### 5. Gotchas
- **`omit_expired=True` is backend-specific.** `SqliteStore` 3.1.1 ignores it, as Part 4 showed. On SQLite,
  either sweep on a schedule (`sweep_interval_minutes`, `start_ttl_sweeper()`), or check an item's age
  yourself (`updated_at`) when even a short stale read is unacceptable
- **`default_ttl` is in minutes, not seconds** — `ttl=30` means 30 *minutes*, which surprises people coming
  from Redis-style TTL-in-seconds APIs
- **`refresh_on_read` changes the whole session model.** `True` gives sliding expiry (any activity extends the
  visit); `False` gives a fixed-length session regardless of activity. Picking the wrong one silently changes
  your security/UX assumptions
- **Session vs. thread key confusion.** Using `thread_id` as the store's namespace key by mistake would make
  "session" memory collapse back into short-term memory
- **Session state must not silently become long-term memory.** Writing the same key into a
  `("semantic", user_id)` namespace must be a deliberate, separate write (see
  `03_long_term_semantic_memory_sqlite.ipynb`), not an accident of a session entry that nobody ever sweeps
- **`isolation_level=None` on the sqlite3 connection is required.** `SqliteStore` manages its own
  `BEGIN`/`COMMIT`; the default DBAPI auto-transaction behavior conflicts with it and raises
  `cannot start a transaction within a transaction`
- Trusted ids and SQLite concurrency apply to every layer: see [`memory_layers_guide.md`](../assets/memory_layers_guide.md)

### Next Steps
- **`03_long_term_semantic_memory_sqlite.ipynb`**: durable facts about a user, available in every thread